# 01 — Exploratory Data Analysis
**RAVDESS Vocal Emotion Recognition**

Dataset: Ryerson Audio-Visual Database of Emotional Speech and Song (Song subset)
- 24 actors (12 male, 12 female)
- 6 emotions: neutral, calm, happy, sad, angry, fearful
- ~1056 audio clips at 22050 Hz

In [ ]:
import sys, os
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
from pathlib import Path

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

DATA_DIR = Path('../../Audio_Song_Actors_01-24')
EMOTION_MAP = {'01':'neutral','02':'calm','03':'happy','04':'sad','05':'angry','06':'fearful'}
EMOTION_NAMES = list(EMOTION_MAP.values())
print('Data dir exists:', DATA_DIR.exists())

## 1. Dataset Inventory

In [ ]:
records = []
for wav in DATA_DIR.rglob('*.wav'):
    parts = wav.stem.split('-')
    if len(parts) != 7: continue
    _, _, emotion, intensity, statement, rep, actor = parts
    if emotion not in EMOTION_MAP: continue
    records.append({
        'file': str(wav),
        'emotion': EMOTION_MAP[emotion],
        'emotion_id': int(emotion),
        'intensity': 'normal' if intensity=='01' else 'strong',
        'actor': int(actor),
        'gender': 'female' if int(actor)%2==0 else 'male'
    })

df = pd.DataFrame(records)
print(f'Total clips: {len(df)}')
print(f'Actors: {df.actor.nunique()}')
df.head()

## 2. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Overall emotion counts
counts = df['emotion'].value_counts().reindex(EMOTION_NAMES)
axes[0].bar(EMOTION_NAMES, counts, color=sns.color_palette('Set2', 6))
axes[0].set_title('Clips per Emotion')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

# Emotion × Gender
ct = df.groupby(['emotion','gender']).size().unstack()
ct.reindex(EMOTION_NAMES).plot(kind='bar', ax=axes[1], color=['#FF9999','#66B2FF'])
axes[1].set_title('Clips per Emotion × Gender')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(title='Gender')

plt.tight_layout()
plt.savefig('../results/eda_class_distribution.png', dpi=150)
plt.show()

## 3. Waveform Visualization

In [ ]:
SR = 22050
fig, axes = plt.subplots(2, 3, figsize=(15, 6))
axes = axes.flatten()

for i, (emo_id, emo_name) in enumerate(EMOTION_MAP.items()):
    sample = df[df.emotion == emo_name].iloc[0]
    y, _ = librosa.load(sample.file, sr=SR, duration=4.0)
    axes[i].plot(np.linspace(0, len(y)/SR, len(y)), y, alpha=0.8, linewidth=0.5)
    axes[i].set_title(f'{emo_name.capitalize()} (actor {sample.actor})')
    axes[i].set_xlabel('Time (s)')
    axes[i].set_ylabel('Amplitude')

plt.suptitle('Waveforms per Emotion', fontsize=14)
plt.tight_layout()
plt.savefig('../results/eda_waveforms.png', dpi=150)
plt.show()

## 4. Mel-Spectrogram Visualization

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
axes = axes.flatten()

for i, (emo_id, emo_name) in enumerate(EMOTION_MAP.items()):
    sample = df[df.emotion == emo_name].iloc[0]
    y, _ = librosa.load(sample.file, sr=SR, duration=4.0)
    mel = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=128, n_fft=2048, hop_length=512)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    img = librosa.display.specshow(mel_db, sr=SR, hop_length=512,
                                    x_axis='time', y_axis='mel', ax=axes[i])
    axes[i].set_title(f'{emo_name.capitalize()}')
    fig.colorbar(img, ax=axes[i], format='%+2.0f dB')

plt.suptitle('Mel-Spectrograms per Emotion', fontsize=14)
plt.tight_layout()
plt.savefig('../results/eda_mel_spectrograms.png', dpi=150)
plt.show()

## 5. MFCC Visualization

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
axes = axes.flatten()

for i, (emo_id, emo_name) in enumerate(EMOTION_MAP.items()):
    sample = df[df.emotion == emo_name].iloc[0]
    y, _ = librosa.load(sample.file, sr=SR, duration=4.0)
    mfcc = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=40, n_fft=2048, hop_length=512)
    img = librosa.display.specshow(mfcc, sr=SR, hop_length=512, x_axis='time', ax=axes[i])
    axes[i].set_title(f'MFCC — {emo_name.capitalize()}')
    fig.colorbar(img, ax=axes[i])

plt.suptitle('MFCCs per Emotion', fontsize=14)
plt.tight_layout()
plt.savefig('../results/eda_mfcc.png', dpi=150)
plt.show()

## 6. Average MFCC Energy per Emotion

In [ ]:
from tqdm.notebook import tqdm

emo_mfccs = {e: [] for e in EMOTION_NAMES}
for _, row in tqdm(df.iterrows(), total=len(df), desc='Computing MFCCs'):
    y, _ = librosa.load(row.file, sr=SR, duration=4.0)
    mfcc = librosa.feature.mfcc(y=y, sr=SR, n_mfcc=40, n_fft=2048, hop_length=512)
    emo_mfccs[row.emotion].append(mfcc.mean(axis=1))

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(40)
for emo, color in zip(EMOTION_NAMES, sns.color_palette('Set2', 6)):
    means = np.stack(emo_mfccs[emo]).mean(axis=0)
    ax.plot(x, means, label=emo, color=color, linewidth=2)

ax.set_xlabel('MFCC Coefficient')
ax.set_ylabel('Mean Value')
ax.set_title('Average MFCC Coefficients per Emotion')
ax.legend(ncol=3)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../results/eda_mfcc_means.png', dpi=150)
plt.show()

## 7. Summary Statistics

In [ ]:
print('=== Dataset Summary ===')
print(df.groupby('emotion').agg(count=('file','count'), 
                                 intensity_dist=('intensity', lambda x: x.value_counts().to_dict())).to_string())
print(f'\nTotal: {len(df)} clips')
print(f'Test set (actors 21-24): {len(df[df.actor.isin([21,22,23,24])])} clips')
print(f'Train+Val  (actors 1-20): {len(df[~df.actor.isin([21,22,23,24])])} clips')